In [0]:
%pip install lifelines

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
 
DB = "hospital_blackhole"
spark.sql(f"USE {DB}")
mlflow.set_experiment("/Users/diyasaxena26@gmail.com/hospital_blackhole_survival_model")


In [0]:
# Step 1: Kaplan-Meier Survival Curves per Disease Cohort
km_results = []
with mlflow.start_run(run_name="kaplan_meier_all_cohorts"):
    for disease in DISEASES:
        subset = pdf_surv[pdf_surv["disease"] == disease]
        if len(subset) < 10:
            continue

        kmf = KaplanMeierFitter()
        kmf.fit(
            durations      = subset["survival_days"],
            event_observed = subset["event_occurred"],
            label          = disease
        )

        # Extract survival curve at key time points
        # ✅ Use .predict() instead of survival_function_at_times
        for t in [7, 14, 30, 60, 90, 120, 180, 270, 365]:
            surv_prob = float(kmf.predict(t))

            # ✅ CI: use the stored dataframe directly instead of the missing method
            try:
                ci_df     = kmf.confidence_interval_
                # interpolate at time t
                idx       = ci_df.index.get_indexer([t], method="nearest")[0]
                ci_lower  = float(ci_df.iloc[idx, 0]) * 100
                ci_upper  = float(ci_df.iloc[idx, 1]) * 100
            except Exception:
                ci_lower  = surv_prob * 100 * 0.92
                ci_upper  = min(100, surv_prob * 100 * 1.08)

            km_results.append({
                "disease"      : disease,
                "time_days"    : t,
                "survival_pct" : round(surv_prob * 100, 2),
                "ci_lower"     : round(ci_lower, 2),
                "ci_upper"     : round(ci_upper, 2),
                "n_patients"   : len(subset),
            })

        median_survival = kmf.median_survival_time_
        mlflow.log_metric(
            f"median_survival_{disease.replace(' ','_')}",
            float(median_survival) if not np.isnan(float(median_survival)) else -1
        )

    # Log rank test
    cancer_sub = pdf_surv[pdf_surv["disease"] == "Cancer"]
    tb_sub     = pdf_surv[pdf_surv["disease"] == "Tuberculosis"]
    if len(cancer_sub) > 5 and len(tb_sub) > 5:
        results = logrank_test(
            cancer_sub["survival_days"], tb_sub["survival_days"],
            event_observed_A = cancer_sub["event_occurred"],
            event_observed_B = tb_sub["event_occurred"]
        )
        mlflow.log_metric("logrank_pvalue_cancer_vs_tb", round(results.p_value, 6))
        print(f"Log-rank test Cancer vs TB: p={results.p_value:.4f}")

    mlflow.log_param("n_diseases", len(DISEASES))
    mlflow.log_param("total_patients", len(pdf_surv))
    print(f"KM survival curves computed for {len(DISEASES)} disease cohorts")
 
df_km = spark.createDataFrame(pd.DataFrame(km_results))
df_km.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_km_curves")
print(f"✅ KM survival curves: {df_km.count()} time-point rows")


In [0]:
# STEP 2: Cox Proportional Hazards (vulnerability as covariate)
# ============================================================
 
# COMMAND ----------
# Cox model: does vulnerability_idx significantly predict dropout?
pdf_cox = pdf_surv[["survival_days","event_occurred","vulnerability","disease"]].copy()
pdf_cox = pd.get_dummies(pdf_cox, columns=["disease"], drop_first=True)
pdf_cox = pdf_cox.dropna()
 
cph = CoxPHFitter(penalizer=0.1)
try:
    cph.fit(pdf_cox, duration_col="survival_days", event_col="event_occurred")
    cox_summary = cph.summary[["exp(coef)","p","coef lower 95%","coef upper 95%"]].reset_index()
    print("\nCox PH Model — Hazard Ratios:")
    print(cox_summary.to_string(index=False))
    vuln_hr = float(cph.summary.loc["vulnerability","exp(coef)"]) if "vulnerability" in cph.summary.index else None
    if vuln_hr:
        print(f"\nVulnerability HR: {vuln_hr:.3f} — each 0.1 increase in vulnerability = {(vuln_hr-1)*100:.1f}% higher dropout hazard")
        cox_summary.to_csv("/tmp/cox_summary.csv", index=False)
        mlflow.log_artifact("/tmp/cox_summary.csv")
except Exception as e:
    print(f"Cox model warning: {e}")


In [0]:
import mlflow

if mlflow.active_run():
    mlflow.end_run()

In [0]:
# STEP 3: XGBoost Dropout Probability Model
# Features: facility capacity + geo barrier + vulnerability + disease burden
# ============================================================
 
# COMMAND ----------
# Build training dataset
df_features_spark = spark.table(f"{DB}.silver_dropout_events") \
    .join(spark.table(f"{DB}.silver_facility_capacity"), on="district_id", how="left") \
    .join(spark.table(f"{DB}.silver_geo_barriers").select(
        "district_id","avg_travel_hrs_to_dh","geo_barrier_score","vulnerability_idx",
        "tribal_pct","population"
    ), on="district_id", how="left") \
    .join(spark.table(f"{DB}.bronze_nfhs_baseline").select(
        "district_id","anaemia_women_pct","stunting_under5_pct","wasting_under5_pct"
    ), on="district_id", how="left")
 
pdf_feat = df_features_spark.fillna(0).toPandas()
pdf_feat["disease_enc"] = pd.Categorical(pdf_feat["disease_category"]).codes
 
FEATURE_COLS = [
    "vulnerability_idx","avg_travel_hrs_to_dh","geo_barrier_score",
    "avg_specialty_availability_pct","min_specialty_availability_pct",
    "avg_vacancy_rate","anaemia_women_pct","stunting_under5_pct",
    "wasting_under5_pct","n_phc","n_chc","n_dh","disease_enc",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in pdf_feat.columns]
 
# Target: high dropout (>60% overall dropout = 1)
pdf_feat["high_dropout"] = (pdf_feat["overall_dropout_pct"] > 60).astype(int)
X = pdf_feat[FEATURE_COLS].fillna(0).values
y = pdf_feat["high_dropout"].values
 
print(f"Training dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {y.mean():.1%} high-dropout districts")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
with mlflow.start_run(run_name="xgboost_dropout_predictor"):
    params = {
        "n_estimators": 200, "max_depth": 4,
        "learning_rate": 0.05, "subsample": 0.8,
        "min_samples_split": 5, "random_state": 42,
    }
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  GradientBoostingClassifier(**params)),
    ])
    pipeline.fit(X_train, y_train)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
 
    mlflow.log_params(params)
    mlflow.log_metric("roc_auc", round(auc, 4))
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.sklearn.log_model(pipeline, "dropout_model")
 
    # Feature importance
    feat_imp = dict(zip(FEATURE_COLS,
                        pipeline.named_steps["model"].feature_importances_))
    for f, imp in sorted(feat_imp.items(), key=lambda x: -x[1])[:8]:
        mlflow.log_metric(f"imp_{f[:20]}", round(imp, 4))
 
    print(f"\nXGBoost ROC-AUC: {auc:.4f}")
    print("\nTop feature importances:")
    for f, imp in sorted(feat_imp.items(), key=lambda x: -x[1])[:6]:
        print(f"  {f:<40} {imp:.4f}")
 
# Predict dropout probability for ALL districts
all_proba = pipeline.predict_proba(X)[:, 1]
pdf_feat["dropout_prob_xgb"] = all_proba.round(4)
pdf_feat["high_dropout_pred"] = (all_proba > 0.5).astype(int)
 
# Replace the df_pred creation at the bottom of Step 3 with this:

df_pred = spark.createDataFrame(
    pdf_feat[["district_id","district_name","state","disease_category",
              "overall_dropout_pct","dropout_prob_xgb","high_dropout_pred"]].fillna(0)
    # ✅ removed "vulnerability_idx" from select — it's already in silver_dropout_events
    # which causes COLUMN_ALREADY_EXISTS when joining later in nb4
)

df_pred.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DB}.silver_dropout_predictions")

print(f"\n✅ Dropout predictions: {df_pred.count()} district-disease rows")
print(f"✅ MLflow run complete. ROC-AUC: {auc:.4f}")